In [2]:
!pip install split-folders


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\dzama\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
!pip install deepface

   ---------------------------------------- 0.0/1.9 MB ? eta -:--:--
   ---------------- ----------------------- 0.8/1.9 MB 3.7 MB/s eta 0:00:01
   --------------------------- ------------ 1.3/1.9 MB 3.0 MB/s eta 0:00:01
   ---------------------------------------- 1.9/1.9 MB 3.2 MB/s  0:00:00

   --- ------------------------------------  1/11 [gunicorn]
   --- ------------------------------------  1/11 [gunicorn]
   --- ------------------------------------  1/11 [gunicorn]
   --- ------------------------------------  1/11 [gunicorn]
   --- ------------------------------------  1/11 [gunicorn]
   --- ------------------------------------  1/11 [gunicorn]
   --- ------------------------------------  1/11 [gunicorn]
   --- ------------------------------------  1/11 [gunicorn]
   --- ------------------------------------  1/11 [gunicorn]
   --- ------------------------------------  1/11 [gunicorn]
   --- ------------------------------------  1/11 [gunicorn]
   --- ---------------------------


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\dzama\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
import os
import shutil
import cv2
import mediapipe as mp
import splitfolders
from deepface import DeepFace


mp_face = mp.solutions.face_detection

face_detector = mp_face.FaceDetection(
    min_detection_confidence=0.5
)


TARGET_EMOTION = {
    "fear": "Fear",
    "surprise": "Surprise",
    "disgust": "Disgust",
    "sad": "Sadness",
    "angry": "Anger"
}


def buat_folder(output_folder):
    os.makedirs(output_folder, exist_ok=True)

    for folder in TARGET_EMOTION.values():
        os.makedirs(
            os.path.join(output_folder, folder),
            exist_ok=True
        )

    os.makedirs(
        os.path.join(output_folder, "Other"),
        exist_ok=True
    )


def crop_wajah(image):
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    hasil = face_detector.process(rgb)

    if not hasil.detections:
        return None

    tinggi, lebar, _ = image.shape

    wajah_terbesar = None
    luas_maksimum = 0

    for deteksi in hasil.detections:

        bbox = deteksi.location_data.relative_bounding_box
        luas = bbox.width * bbox.height

        if luas > luas_maksimum:
            luas_maksimum = luas
            wajah_terbesar = bbox

    x = int(wajah_terbesar.xmin * lebar)
    y = int(wajah_terbesar.ymin * tinggi)
    w = int(wajah_terbesar.width * lebar)
    h = int(wajah_terbesar.height * tinggi)

    x = max(0, x)
    y = max(0, y)

    w = min(lebar - x, w)
    h = min(tinggi - y, h)

    wajah = image[y:y+h, x:x+w]

    if wajah.size == 0:
        return None

    return cv2.resize(wajah, (224, 224))


def prediksi_emosi(wajah):

    try:

        hasil = DeepFace.analyze(
            wajah,
            actions=["emotion"],
            enforce_detection=False,
            silent=True
        )

        if isinstance(hasil, list):
            hasil = hasil[0]

        return hasil["dominant_emotion"]

    except Exception:
        return None


def simpan_gambar(gambar, emosi, nama_file, folder_output):

    if emosi in TARGET_EMOTION:
        tujuan = TARGET_EMOTION[emosi]
    else:
        tujuan = "Other"

    lokasi = os.path.join(
        folder_output,
        tujuan,
        nama_file
    )

    cv2.imwrite(lokasi, gambar)


def proses_dataset(input_folder, output_folder):

    jumlah = 0

    for root, _, files in os.walk(input_folder):

        nama_video = os.path.basename(root)

        for file in files:

            if not file.lower().endswith((".jpg", ".png")):
                continue

            gambar = cv2.imread(
                os.path.join(root, file)
            )

            if gambar is None:
                continue

            wajah = crop_wajah(gambar)

            if wajah is None:
                continue

            emosi = prediksi_emosi(wajah)

            simpan_gambar(
                wajah,
                emosi,
                f"{nama_video}_{file}",
                output_folder
            )

            jumlah += 1

    print(f"Total gambar diproses : {jumlah}")


def hapus_folder_other(folder_output):

    other = os.path.join(folder_output, "Other")

    if os.path.exists(other):
        shutil.rmtree(other)


def split_dataset(folder_input, folder_output):

    splitfolders.ratio(
        folder_input,
        output=folder_output,
        seed=42,
        ratio=(0.7, 0.2, 0.1),
        group_prefix=None,
        move=False
    )


def jalankan_pipeline_total(
    input_base="dataset_frame",
    labeled_base="dataset_labeled",
    final_base="dataset_final"
):

    buat_folder(labeled_base)

    proses_dataset(
        input_base,
        labeled_base
    )

    hapus_folder_other(
        labeled_base
    )

    split_dataset(
        labeled_base,
        final_base
    )

    print("Pipeline selesai.")


if __name__ == "__main__":
    jalankan_pipeline_total()


26-07-03 22:12:41 - Directory C:\Users\dzama\.deepface has been created
26-07-03 22:12:41 - Directory C:\Users\dzama\.deepface\weights has been created
26-07-03 22:12:46 - 🔗 facial_expression_model_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/facial_expression_model_weights.h5 to C:\Users\dzama\.deepface\weights\facial_expression_model_weights.h5...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/facial_expression_model_weights.h5
To: C:\Users\dzama\.deepface\weights\facial_expression_model_weights.h5
100%|██████████| 5.98M/5.98M [00:00<00:00, 7.29MB/s]


Total gambar diproses : 2213


Copying files: 872 files [00:16, 52.88 files/s]

Pipeline selesai.
